# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring.**

I'm choosing this over other lanes because it's the only lane whose output is something a person acts on directly: a ranked queue of pages with a reason code and confidence label attached, not just a report or a set of labeled groups. It also forces me to build the full pipeline end-to-end — a rule-based baseline, an optional model that has to beat that baseline, and honest validation against a real outcome — rather than stopping at analysis. That's the version of this work most useful to actually take further, and the one where getting it wrong (leakage, a fake signal, an overclaimed causal story) is easiest to catch early if I'm careful.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision this work improves:** Out of thousands of content pages, which ones should get reviewed first — this week, given limited editor time.

**Who acts on it, and what they do:** A content editor or SEO lead opens the top of the ranked queue and applies the suggested treatment to each page — refresh, expand, protect, prune, or monitor.

**What a wrong recommendation costs:** Two different failure directions, and they don't cost the same.
- **False positive** — a page gets flagged as worth fixing but wasn't. Cost: wasted editor hours, which is the scarcer resource in this system.
- **False negative** — a genuinely declining page never surfaces in the queue. Cost: the page keeps losing traffic silently, unnoticed until it's worse and harder to recover.

Because editor time is the bottleneck, I'll weight precision at the top of the queue more heavily than recall — a few missed decliners further down the list is a cheaper mistake than sending an editor to fix pages that didn't need it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("/workspaces/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Sanity check the load
print(f"Shape: {df.shape}")
print(f"Clients: {df['client_id'].nunique()}")

# 1. Scale — how many pages are flagged declining?
declining = (df['trend_direction'] == 'down').sum()
print(f"\n1. Declining: {declining} / {len(df)} = {declining/len(df):.1%}")

# 2. Does visibility concentrate decline, or is it spread evenly?
visible = df[df['position_tier'].isin(['top_3', 'page_1'])]
visible_rate = (visible['trend_direction'] == 'down').mean()
print(f"2. Visible pages (top_3/page_1): {len(visible)} pages, {visible_rate:.1%} declining")

# 3. Does staleness track with decline?
df['is_declining'] = df['trend_direction'] == 'down'
staleness = df.groupby(pd.cut(df['days_since_last_update'], [0,30,90,180,365,99999]))['is_declining'].mean()
print(f"\n3. Decline rate by staleness bucket:\n{staleness}")

Shape: (30000, 44)
Clients: 32

1. Declining: 16262 / 30000 = 54.2%
2. Visible pages (top_3/page_1): 14135 pages, 51.6% declining

3. Decline rate by staleness bucket:
days_since_last_update
(0, 30]         0.511377
(30, 90]        0.588571
(90, 180]       0.611057
(180, 365]      0.467456
(365, 99999]    0.600000
Name: is_declining, dtype: float64


**1. Scale:** 16,262 / 30,000 pages (54.2%) are flagged as declining. This is a large enough share that triage is genuinely needed — but a ~coin-flip split across 30k rows is also a flag on its own: with "declining" defined as a >20% drop over a single 30-day window, some of this is likely short-term noise (seasonality, campaign timing) rather than real content failure. Worth stating plainly rather than treating the number as pure validation.

**2. Visibility doesn't concentrate decline:** 14,135 pages sit in top_3/page_1 position tiers, and 51.6% of them are declining — almost identical to the 54.2% overall rate. Visible, high-value pages aren't disproportionately the ones at risk, which means a ranking that leans on position tier alone won't separate what matters from what doesn't.

**3. Staleness alone doesn't explain decline:** decline rate by `days_since_last_update` bucket is 51.1% → 58.9% → 61.1% → 46.7% → 60.0% — not a clean, monotonic trend. If "outdated content declines more" were simply true, this would climb steadily. It doesn't, which rules out a single-threshold rule ("flag anything >90 days stale") and is itself the argument for scoring based on multiple interacting signals rather than one rule doing the work.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work CAN say:**
- This is a **decision-support ranking**, not a forecast — it prioritizes which pages an editor should review first, based on patterns in the trailing 90-day window.
- The score is **directional and comparative**: "this page looks riskier than that one right now," not a prediction of future traffic.
- Any pattern reported (staleness, content type, position) is an **observed association** in this dataset, for these 32 clients, over this window — not a universal SEO rule.
- Accuracy figures like precision@K describe how well the ranking matched outcomes in a **held-out, time-separated sample of past data** — a backward-looking measure, not a promise about the future.

**What this work CANNOT say:**
- **Not causal.** I can't claim refreshing a flagged page *will cause* it to recover — correlation between staleness and decline isn't even clean in this data (section 3), let alone proof of cause.
- **Not a Google prediction.** This scores internal signals (traffic, position, staleness) against a client's own history — it says nothing about why Google's algorithm ranks a page where it does.
- **Not guaranteed ROI.** I can't promise fixing the top 20 pages recovers a specific amount of traffic — that would require an actual experiment (refresh + control group), which is out of scope here.
- **Not client-agnostic.** A pattern holding across these 32 clients may not hold for a new client; I shouldn't imply the model generalizes universally.
- **Not free of the 54% base-rate problem.** Since "declining" is close to a coin flip in this slice, some of that signal may be noise from a short comparison window — the tool's confidence score shouldn't imply more certainty than the label actually earns.

**Summary:** This system produces a prioritized, explainable review queue — it identifies where attention is statistically warranted, not where success is guaranteed. It observes correlation in past data; it does not establish causation, and it does not model Google's ranking behavior.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.